# ArchRivet: optional Qwen 4B QLoRA pilot
This notebook is prepared for a user-connected Colab GPU runtime; cloud training has not been executed or validated by this repository. Select a GPU in Runtime → Change runtime type. No paid runtime is rented by this code.

The experiment uses the frozen synthetic backend diagnostic dataset. It does not open the final test split, certify production quality, or replace the plugin's current model. Training uses NF4, rank 8, alpha 16, dropout 0.05, all linear layers, learning rate 1e-4, batch 1, accumulation 16, 1024 tokens, and at most two epochs. Only adapters and reports are copied to your selected Drive folder.

Drive storage does not increase runtime scratch space. [Colab resource and storage guidance](https://research.google.com/colaboratory/faq.html). [Qwen model](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [PEFT quantization guide](https://huggingface.co/docs/peft/developer_guides/quantization).

In [ ]:
# Run this before cloning, installing dependencies, or downloading any model.
import shutil
import sys
from pathlib import Path
assert (3, 11) <= sys.version_info[:2] < (3, 13), 'This project currently supports Python 3.11 and 3.12.'
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime first. No downloads have started.'
SCRATCH = Path('/content/archrivet-work')
assert shutil.disk_usage('/content').free >= 30 * 1024**3, 'At least 30 GiB free runtime scratch is required.'
SCRATCH.mkdir(exist_ok=True)
print({'gpu': torch.cuda.get_device_name(0), 'free_runtime_gib': round(shutil.disk_usage(SCRATCH).free / 1024**3, 1)})


In [ ]:
import subprocess
import sys
import os
REPO = Path('/content/archrivet')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ujjansh05/archrivet.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-qlora.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
print('Repository revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
# Start with seed 17; repeat separately with 29 after inspecting the first report.
SEED = 17
EPOCHS = 1
TIMEOUT_SECONDS = 3600
HOURLY_RATE_USD = 0.0  # Actual runtime rate, not an assumed discount.
RUN_TRAINING = False  # Set True only after the dry-run below passes.
UPLOAD_TO_DRIVE = False  # Set True to authenticate and copy this run's artifacts.
DRIVE_FOLDER_ID = ''  # Paste only the intended folder ID here; never commit this value.
OUTPUT = SCRATCH / f'qwen4b-qlora-seed{SEED}-pilot1'
COMMAND = [sys.executable, 'scripts/finetune_qlora.py', '--data', 'datasets/backend-diagnostics-v1',
           '--scratch', str(SCRATCH), '--output', str(OUTPUT), '--seed', str(SEED), '--epochs', str(EPOCHS),
           '--timeout-seconds', str(TIMEOUT_SECONDS), '--hourly-rate-usd', str(HOURLY_RATE_USD)]
subprocess.run(COMMAND + ['--dry-run'], check=True)


The dry-run verifies pinned train/development/calibration hashes and disk space without model downloads. Training rechecks CUDA and data before downloads. Inputs exceeding 1024 tokens are rejected without truncation. Loss applies only to the assistant verdict JSON. Development assistant-token loss selects the adapter; calibration predictions are retained but no calibration fit is claimed.

The process deadline does not stop billing for a separately rented machine. Disconnect the runtime when finished. The runtime-local experiment ledger permits at most USD 10 compute within the USD 15 experiment budget, with USD 5 reserved. **Deleting/resetting the Colab runtime loses this ledger: it does not aggregate costs across sessions and is not a global billing cap.** Keep your own cumulative spending record; the script cannot discover or control external charges. No paid runtime is rented automatically.

In [ ]:
if RUN_TRAINING:
    subprocess.run(COMMAND + ['--run'], check=True, timeout=TIMEOUT_SECONDS + 180)
else:
    print('Dry-run only. Set RUN_TRAINING=True when ready for the bounded GPU pilot.')


In [ ]:
import json
if (OUTPUT / 'report.json').exists():
    report = json.loads((OUTPUT / 'report.json').read_text())
    print({'before': report['before_development']['overall'], 'after': report['after_development']['overall'],
           'final_test_used': report['final_test_used'], 'quality_release_approved': report['quality_release_approved']})
elif OUTPUT.exists():
    print('Run has no completed report. Inspect run.json and training.log locally.')


In [ ]:
# No Drive mount. OAuth credentials stay in the runtime and are never printed or uploaded.
if UPLOAD_TO_DRIVE:
    sys.path.insert(0, str(REPO / 'scripts'))
    from drive_artifacts import artifact_plan, upload_artifacts, validate_folder_id
    validate_folder_id(DRIVE_FOLDER_ID)
    plan = artifact_plan(OUTPUT)
    print({'artifact_count': len(plan), 'total_bytes': sum(item['bytes'] for item in plan)})
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    credentials, _ = google.auth.default()
    drive_service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    result = upload_artifacts(drive_service, OUTPUT, DRIVE_FOLDER_ID, plan=plan)
    print({'uploaded': sum(item['status'] == 'uploaded' for item in result),
           'already_present': sum(item['status'] == 'already_present' for item in result)})
else:
    print('Drive upload disabled. Set a folder ID and UPLOAD_TO_DRIVE=True to copy the selected artifacts.')
